# Denoising Autoencoders with PyTorch

## Lecture Notebook — MNIST Denoising Autoencoder


# 1. What is a Denoising Autoencoder?

An **autoencoder** is a neural network trained to reconstruct its own input.

It consists of two components:

1. **Encoder**
   Compresses the noisy input into a low-dimensional representation.

2. **Decoder**
   Reconstructs the denoised input from the compressed representation.

Pipeline:

$x \rightarrow \text{Encoder} \rightarrow z \rightarrow \text{Decoder} \rightarrow \hat{x}$

Where:

* $x$ = noisy image
* $z$ = latent vector (compressed representation)
* $\hat{x}$ = clean image

The model learns to map:

$$ \text{Noisy Image} \rightarrow \text{Clean Image} $$

This forces the latent representation to capture meaningful geometric structure rather than just memorizing the input.

# 2. Imports


In [ ]:
import copy
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from torchvision.datasets import MNIST


# 3. Reproducibility and Device Setup

Machine learning experiments should be reproducible.

We seed:

* Python random
* NumPy
* PyTorch

In [ ]:
def seed_everything(seed=42):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()


# 4. Device Selection

We automatically choose:

* CUDA GPU if available
* Apple Metal (MPS) if available
* CPU otherwise


In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Using device:", device)


# 5. Loading the MNIST Dataset

MNIST contains:

* 60,000 training images
* 10,000 test images
* Grayscale handwritten digits
* Image size: (28 \times 28)


In [ ]:
## Convert Images to Tensors
transform = transforms.ToTensor()
## Download Dataset


In [ ]:
mnist_train_full = MNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True
)

mnist_test = MNIST(
    root="./data",
    train=False,
    transform=transform,
    download=True
)

# 6. Adding Gaussian Noise

Gaussian Noise Formula

$$ x_{\text{noisy}} = x + \epsilon, \quad \epsilon \sim \mathcal{N}(0, \sigma^2) $$


In [ ]:
def add_noise(x, noise_factor=0.5):
    noise = torch.randn_like(x) * noise_factor
    x_noisy = x + noise
    return torch.clamp(x_noisy, 0., 1.)


In [ ]:
images = torch.stack([mnist_test[i][0] for i in range(8)])

fig, axes = plt.subplots(2, 8, figsize=(12, 4))

for i in range(8):
    # Original
    axes[0, i].imshow(images[i][0].cpu(), cmap="gray")
    axes[0, i].set_xticks([])
    axes[0, i].set_yticks([])

    # Reconstruction
    axes[1, i].imshow(add_noise(images)[i][0].cpu(), cmap="gray")
    axes[1, i].set_xticks([])
    axes[1, i].set_yticks([])

# Labels
axes[0, 0].set_ylabel("Original", fontsize=12)
axes[1, 0].set_ylabel("Noisy", fontsize=12)

# Remove borders/spines
for ax in axes.flatten():
    for spine in ax.spines.values():
        spine.set_visible(False)

plt.tight_layout()
plt.show()    

# 7. Train / Validation Split

We split the original training set into:

* 55,000 training images
* 5,000 validation images


In [ ]:
mnist_train, mnist_val = random_split(
    mnist_train_full,
    [55000, 5000],
    generator=torch.Generator().manual_seed(42)
)


# 8. DataLoaders

DataLoaders help us:

* Create mini-batches
* Shuffle data
* Load data efficiently

In [ ]:
batch_size = 128

train_loader = DataLoader(
    mnist_train,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    mnist_val,
    batch_size=batch_size,
    shuffle=False
)


# 9. Convolutional Autoencoder

Our architecture follows this pipeline:
```
Input Image (1×28×28)
        ↓
      Conv2D
        ↓
      Conv2D
        ↓
Latent Vector (2D)
        ↓
   Linear Layer
        ↓
 ConvTranspose2D
        ↓
 ConvTranspose2D
        ↓
Reconstructed Image
```

The encoder compresses the image into a tiny latent representation, while the decoder reconstructs the image from that compressed encoding.


In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        
        # ---------------- ENCODER ----------------
        self.encoder = nn.Sequential(
            # 28x28 -> 14x14
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1, dilation=1),
            nn.ReLU(),
            # 14x14 -> 7x7
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1, dilation=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, latent_dim)
        )

        # ---------------- DECODER ----------------
        self.decoder_input = nn.Linear(latent_dim, 32 * 7 * 7)
        self.decoder = nn.Sequential(
            # 7x7 -> 14x14
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1, dilation=1),
            nn.ReLU(),
            # 14x14 -> 28x28
            nn.ConvTranspose2d(16, 8, kernel_size=3, stride=2, padding=1, output_padding=1, dilation=1),
            nn.ReLU(),
            # Final channel reduction
            nn.Conv2d(8, 1, kernel_size=3, padding=1, dilation=1),
            nn.Sigmoid()
        )

    def encode(self, x):
        return self.encoder(x)

    def forward(self, x):
        # Encode
        z = self.encode(x)
        # Decode
        x = self.decoder_input(z)
        x = x.view(-1, 32, 7, 7)
        x = self.decoder(x)
        return x

# 10. Loss Function

The model learns to minimize reconstruction error.

We use Mean Squared Error (MSE):
    
$\text{MSE} = \frac{1}{N}\sum_{i=1}^{N}(x_i - \hat{x}_i)^2$

Where:

* $x_i$ = original pixel
* $\hat{x}_i$ = reconstructed pixel

# 11. Optimizer

We use Adam optimization, that combines:

* Momentum
* Adaptive learning rates

to improve convergence speed and stability.


# 12. Early Stopping

Deep learning models may overfit.

Early stopping monitors validation loss and stops training when the model stops improving.


In [ ]:
class EarlyStopping:
    def __init__(self, patience=5):
        self.patience = patience
        self.best_loss = float("inf")
        self.counter = 0
        self.best_weights = None
    def step(self, model, val_loss):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.best_weights = copy.deepcopy(model.state_dict())
            self.counter = 0
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience

    def restore(self, model):
        if self.best_weights is not None:
            model.load_state_dict(self.best_weights)


# 13. Training Loop

For every batch:

1. Forward pass
2. Compute loss
3. Backpropagation
4. Update weights

In [ ]:
def train_model(model, train_loader, val_loader, device, epochs=50, learning_rate=1e-3, patience=5):
    model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    early_stopping = EarlyStopping(patience=patience)
    train_losses = []
    val_losses = []
    for epoch in range(epochs):
        # ---------------- TRAIN ----------------
        model.train()
        train_loss = 0
        for images, _ in train_loader:
            images = images.to(device)
            inputs = add_noise(images)
            outputs = model(inputs)
            loss = criterion(outputs, images)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        # ---------------- VALIDATION ----------------
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for images, _ in val_loader:
                images = images.to(device)
                inputs = add_noise(images)
                outputs = model(inputs)
                loss = criterion(outputs, images)
                val_loss += loss.item()

        val_loss /= len(val_loader)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        # ---------------- EARLY STOPPING ----------------
        stop = early_stopping.step(model, val_loss)
        if stop:
            print("\nEarly stopping triggered.")
            break

    # Restore best model weights
    early_stopping.restore(model)
    print(f"\nBest Validation Loss: {early_stopping.best_loss:.4f}")
    return train_losses, val_losses


In [ ]:
# Initialize model
model = Autoencoder(latent_dim=2)

In [ ]:
train_losses, val_losses = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device, 
    epochs=100,
    learning_rate=1e-3,
    patience=5
)

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid') 
plt.figure(figsize=(10, 5), dpi=100)

# Plot training and validation curves with distinct semantic colors and weights
plt.plot(train_losses, label="Training Loss", color="#2b5c8f", linewidth=2.5)
plt.plot(val_losses, label="Validation Loss", color="#d95f02", linewidth=2.5, linestyle="--")

# Highlight the minimum validation loss point (where Early Stopping saves weights)
best_epoch = val_losses.index(min(val_losses))
best_loss = min(val_losses)
plt.scatter(best_epoch, best_loss, color="#d95f02", edgecolor="black", 
            s=100, zorder=5, label=f"Best Model (Epoch {best_epoch+1})")

# Enhancing text and metadata
plt.title("Autoencoder Training Convergence & Loss Profile", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Training Epochs", fontsize=11, labelpad=10)
plt.ylabel("Mean Squared Error (MSE)", fontsize=11, labelpad=10)

# Refined grid lines for easy readability without visual clutter
plt.grid(True, linestyle=":", alpha=0.6, color="#cccccc")

# Strategic legend placement
plt.legend(loc="upper right", frameon=True, facecolor="white", edgecolor="#e0e0e0", fontsize=10)

plt.tight_layout()
plt.show()

# 17. Reconstruction Visualization

To evaluate reconstruction quality we compare:

* Original image
* Noisy image
* Reconstructed image



In [ ]:
def visualize_reconstruction(model, dataset, device):
    model.eval()

    # Get 8 images
    images = torch.stack([dataset[i][0] for i in range(8)]).to(device)

    # Reconstruct
    with torch.no_grad():
        reconstructed = model(images)

    # Move to CPU
    images = images.cpu()
    reconstructed = reconstructed.cpu()

    fig, axes = plt.subplots(3, 8, figsize=(12, 4))

    for i in range(8):
        # Original
        axes[0, i].imshow(images[i][0].cpu(), cmap="gray")
        axes[0, i].set_xticks([])
        axes[0, i].set_yticks([])
        # Noisy
        axes[1, i].imshow(add_noise(images)[i][0].cpu(), cmap="gray")
        axes[1, i].set_xticks([])
        axes[1, i].set_yticks([])
        # Reconstruction
        axes[2, i].imshow(reconstructed[i][0].cpu(), cmap="gray")
        axes[2, i].set_xticks([])
        axes[2, i].set_yticks([])

    # Labels
    axes[0, 0].set_ylabel("Original", fontsize=12)
    axes[1, 0].set_ylabel("Noisy", fontsize=12)
    axes[2, 0].set_ylabel("Denoised", fontsize=12)

    # Remove borders/spines
    for ax in axes.flatten():
        for spine in ax.spines.values():
            spine.set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
visualize_reconstruction(model, mnist_test, device)

# 18. Latent Space Visualization

Since the latent dimension is 2, we can directly visualize encoded images in 2D space.


In [ ]:
def visualize_latent_space(model, dataset, num_points=2000):
    model.eval()
    latent_vectors = []
    labels = []
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            z = model.encode(images)
            latent_vectors.append(z.cpu())
            labels.append(targets)

    latent_vectors = torch.cat(latent_vectors)[:num_points]
    labels = torch.cat(labels)[:num_points]
    
    x = latent_vectors[:, 0].numpy()
    y = latent_vectors[:, 1].numpy()
    
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(x, y, c=labels, cmap="tab10", alpha=0.7)
    plt.colorbar(scatter, label="Digit Class")
    plt.title("2D Latent Space Map")
    plt.xlabel("Latent Dimension 1")
    plt.ylabel("Latent Dimension 2")
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
visualize_latent_space(model, mnist_test)


# 19. Latent Space Traversal (Generative Inference)

Because our model compresses all images down into a continuous 2D space, we can map out a specific continuous coordinate line (trajectory) in that latent space.

By passing coordinates along this trajectory line directly to our **Decoder**, we can see how the decoder generates new synthetic pixel-space data that morphs smoothly between hand-written digits.


In [ ]:
def plot_latent_traversal_with_map(model, dataset, start_point=(-2.0, -1.5), end_point=(2.0, 1.5), steps=10, num_points=2000):
    """
    Superimposes a continuous traversal line on top of the 2D scatter map of dataset classes (Left),
    while generating and showing the corresponding decoded outputs from the latent trajectory (Right).
    """
    model.eval()
    
    # 1. Gather latent points from the dataset for the background map
    latent_vectors = []
    labels = []
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            z = model.encode(images)
            latent_vectors.append(z.cpu())
            labels.append(targets)

    latent_vectors = torch.cat(latent_vectors)[:num_points]
    labels = torch.cat(labels)[:num_points]
    
    map_x = latent_vectors[:, 0].numpy()
    map_y = latent_vectors[:, 1].numpy()
    
    # 2. Generate the linear trajectory pathway points
    t = np.linspace(0, 1, steps)
    latent_line = np.array([[(1-val)*start_point[0] + val*end_point[0], 
                             (1-val)*start_point[1] + val*end_point[1]] for val in t])
    
    latent_tensor = torch.tensor(latent_line, dtype=torch.float32).to(device)
    
    # 3. Decode the trajectory points into generative images
    with torch.no_grad():
        x_expanded = model.decoder_input(latent_tensor)
        x_reshaped = x_expanded.view(-1, 32, 7, 7)
        generated_images = model.decoder(x_reshaped).cpu().squeeze(1).numpy()
        
    # 4. Create the side-by-side visualization layout
    fig = plt.figure(figsize=(16, 6), dpi=100)
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.3])
    
    # --- LEFT SUBPLOT: Superimposed Latent Map & Line ---
    ax_latent = fig.add_subplot(gs[0])
    
    # Draw background cluster points (slightly transparent to keep trajectory clear)
    scatter = ax_latent.scatter(map_x, map_y, c=labels, cmap="tab10", alpha=0.4, s=15, zorder=1)
    cbar = plt.colorbar(scatter, ax=ax_latent, label="Digit Class")
    
    # Draw the continuous red trajectory line
    ax_latent.plot(latent_line[:, 0], latent_line[:, 1], color='#e74c3c', 
                   linewidth=3.5, linestyle='-', zorder=2, label='Traversal Line')
    
    # Draw numbered markers on the trajectory line
    ax_latent.scatter(latent_line[:, 0], latent_line[:, 1], color='#2c3e50', 
                      edgecolor='white', s=80, zorder=3)
    
    # Label the step numbers (S-1, S-2, etc.) right on the map
    for i, pt in enumerate(latent_line):
        ax_latent.text(pt[0] + 0.12, pt[1] - 0.05, f"S-{i+1}", fontsize=9, 
                       weight='bold', color='black', bbox=dict(facecolor='white', alpha=0.8, boxstyle='round,pad=0.2'))
        
    ax_latent.set_title("1. Trajectory Path Across Latent Embedding Map", fontsize=12, fontweight='bold', pad=10)
    ax_latent.set_xlabel("Latent Dimension 1")
    ax_latent.set_ylabel("Latent Dimension 2")
    ax_latent.grid(True, linestyle=":", alpha=0.4)
    ax_latent.legend(loc="upper left", frameon=True, facecolor='white')
    
    # --- RIGHT SUBPLOT: Decoded Real-Space Output Strips ---
    compiled_strip = np.hstack([generated_images[i] for i in range(steps)])
    
    ax_real = fig.add_subplot(gs[1])
    ax_real.imshow(compiled_strip, cmap='gray')
    ax_real.axis('off')
    ax_real.set_title("2. Decoded Generative Space Output ($x$)", fontsize=12, fontweight='bold', pad=10)
    
    # Add step indicators under each image frame
    for i in range(steps):
        ax_real.text(14 + (i * 28), 34, f"S-{i+1}", color='#e74c3c', 
                    fontsize=10, weight='bold', ha='center')
        
    plt.tight_layout()
    plt.show()

In [ ]:
# Run the generation script over an arbitrary structural coordinate vector slice
plot_latent_traversal_with_map(model, mnist_test, start_point=(-5.5, 2.0), end_point=(1.0, 0.0), steps=10)

# 20. Key Takeaways

By the end of this notebook, you should understand:

* How autoencoders compress information.
* Why convolutional layers work well for retaining spatial hierarchies in images.
* How denoising improves representation learning by preventing identity mapping.
